In [ ]:
from matplotlib import pyplot
import opacus.accountants
from opacus.accountants import RDPAccountant, PRVAccountant

delta = 1e-5

def epsilon(noise_multiplier_1, noise_multiplier_2, accountant, m=1):
    filename = f"{type(accountant).__name__}_{noise_multiplier_1:.2f}_{noise_multiplier_2:.2f}_{noise_multiplier_switch}_{rounds}_{batch_size}_{n_samples}_{m}.txt"
    try:
        with open(filename) as file:
            return [float(line) for line in file]
    except FileNotFoundError:
        pass
    n_batches = (n_samples * m + batch_size - 1) // batch_size
    sample_rate = m / n_batches
    epsilons = [0]
    for round_idx in range(rounds):
        noise_multiplier = noise_multiplier_1 if round_idx < noise_multiplier_switch else noise_multiplier_2
        for _ in range(n_batches):
            accountant.step(noise_multiplier=noise_multiplier, sample_rate=sample_rate)
        epsilons.append(float(accountant.get_epsilon(delta=delta)))
    # eps, best_alpha = accountant.get_privacy_spent(delta=delta)
    # print(best_alpha)
    with open(filename, "w") as file:
        for value in epsilons:
            file.write(f"{value}\n")
    return epsilons

def plot_epsilon(data):
    figure = pyplot.figure(figsize=(13,4))
    for name, arr in data.items():
        pyplot.plot(arr, label=name)
    pyplot.xlabel("round")
    pyplot.ylabel("epsilon")
    pyplot.grid(True, alpha=0.3)
    pyplot.legend()
    pyplot.show()
    # figure.savefig("epsilon.pdf", bbox_inches="tight")

##### page break

In [ ]:
rounds=200
noise_multiplier_switch = 180
batch_size = 32
n_samples = 1600 # emotion 16000 training set / 10 clients

data = {}
data["fixed"] = epsilon(1.00, 1.00, PRVAccountant())
data["dpa-dp-0.85"] = epsilon(1.03, 0.85, PRVAccountant())
data["dpa-dp-0.75"] = epsilon(1.08, 0.75, PRVAccountant())
data["dpa-dp-0.65"] = epsilon(1.35, 0.65, PRVAccountant())
print({k: a[-1] for k, a in data.items()})
plot_epsilon(data)

In [ ]:
rounds=200
noise_multiplier_switch = 180
batch_size = 32
n_samples = 1600 # emotion 16000 training set / 10 clients

data = {}
data["fixed"] = epsilon(1.00, 1.00, RDPAccountant())
data["dpa-dp-0.85"] = epsilon(1.03, 0.85, RDPAccountant())
data["dpa-dp-0.75"] = epsilon(1.08, 0.75, RDPAccountant())
data["dpa-dp-0.65"] = epsilon(1.35, 0.65, RDPAccountant())
print({k: a[-1] for k, a in data.items()})
plot_epsilon(data)